# Test model YOLO26 — ảnh & video (FPS)

Bộ test cho 2 model:

- `yolo26s_img640_baseline_v1_small`
- `yolo26s_img640_baseline_v1_nano`

| # | Test | Đầu vào | Đo lường |
|---|------|---------|----------|
| 1 | **Image test** | File ảnh **hoặc thư mục chứa ảnh** | Số bbox, nhãn, độ tin cậy, inference ms, FPS + ảnh dự đoán |
| 2 | **Video test (FPS test)** | File video | FPS từng frame + **biểu đồ FPS dưới mỗi model** + **video mp4 có bbox** + frame mẫu |

`IMAGE_PATH` / `VIDEO_PATH` nhận: đường dẫn local (tương đối hoặc đầy đủ), thư mục ảnh, hoặc URL `http(s)://...`.

In [1]:
import time
import urllib.request
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from ultralytics import YOLO

#0. Cấu hình

In [2]:
MODEL_SMALL = "yolo26s_img640_baseline_v1_small"
MODEL_NANO = "yolo26s_img640_baseline_v1_nano"

MODEL_DIRS = {
    MODEL_SMALL: "yolo26s_img640_baseline_v1_small/yolo26s_img640_baseline_v1_small/weights/best.pt",
    MODEL_NANO: "yolo26s_img640_baseline_v1-nano/yolo26s_img640_baseline_v1-nano/weights/best.pt",
}

IMAGE_PATH = r"/home/loc-nguyen/PROJECT/AI/hive-management-ai/ai_engine/data/bee-dectection/testing_data/test_img.png"
VIDEO_PATH = r"stock-footage-close-up-of-honeybees-warming-in-sun-on-their-beehive-beekeeping-process-honey-productio-and.mp4"

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
IMG_SIZE = 640
CONF_THRESHOLD = 0.25
WARMUP_RUNS = 3
IMAGE_REPEATS = 20
FPS_MAX_FRAMES = 200
SAMPLE_FRAME_STRIDE = 10
MAX_SAMPLE_FRAMES = 6
DEVICE = "0" if torch.cuda.is_available() else "cpu"

In [3]:
def find_weights_dir():
    for base in (Path.cwd(), *Path.cwd().parents):
        for candidate in (base, base / "ai_engine"):
            weights = candidate / "weights" / "dection_bee"
            if weights.is_dir():
                return weights
    raise FileNotFoundError("khong tim thay thu muc weights/dection_bee")


def download_url(url, cache_dir):
    cache_dir.mkdir(parents=True, exist_ok=True)
    target = cache_dir / Path(url.split("?")[0]).name
    if target.is_file() and target.stat().st_size > 0:
        return target
    with urllib.request.urlopen(url, timeout=30) as response:
        target.write_bytes(response.read())
    return target


def media_source(raw_path, label, allow_dir=True):
    if not raw_path or not raw_path.strip():
        print(f"[SKIP] {label} chua duoc dien duong dan.")
        return None
    if raw_path.startswith(("http://", "https://")):
        try:
            path = download_url(raw_path, Path.cwd() / "media_cache")
        except Exception as error:
            print(f"[FAIL] {label} tai ve that bai: {error}")
            return None
        print(f"[PASS] {label} (url) -> {path}")
        return path
    path = Path(raw_path).expanduser()
    if not path.is_absolute():
        path = Path.cwd() / path
    path = path.resolve()
    if not path.exists():
        print(f"[FAIL] {label} khong ton tai: {path}")
        print(f"       cwd hien tai: {Path.cwd()}")
        return None
    if path.is_file():
        print(f"[PASS] {label} -> {path}")
        return path
    if allow_dir and path.is_dir():
        print(f"[PASS] {label} (thu muc) -> {path}")
        return path
    print(f"[FAIL] {label} khong phai file hop le: {path}")
    return None


def collect_images(source):
    if source.is_file():
        return [source]
    return sorted(p for p in source.iterdir() if p.suffix.lower() in IMAGE_SUFFIXES)


def warmup(model, source):
    for _ in range(WARMUP_RUNS):
        model.predict(
            source if isinstance(source, np.ndarray) else str(source),
            imgsz=IMG_SIZE,
            device=DEVICE,
            verbose=False,
        )


def load_models():
    weights_dir = find_weights_dir()
    models = {}
    for name, relative in MODEL_DIRS.items():
        path = weights_dir / relative
        if path.is_file():
            models[name] = YOLO(str(path))
            print(f"[PASS] {name} -> {path}")
        else:
            print(f"[FAIL] {name} -> {path}")
    return models


WEIGHTS_DIR = find_weights_dir()
MODELS = load_models()

[PASS] yolo26s_img640_baseline_v1_small -> /home/loc-nguyen/PROJECT/AI/hive-management-ai/ai_engine/bee-dectection/weights/dection_bee/yolo26s_img640_baseline_v1_small/yolo26s_img640_baseline_v1_small/weights/best.pt
[PASS] yolo26s_img640_baseline_v1_nano -> /home/loc-nguyen/PROJECT/AI/hive-management-ai/ai_engine/bee-dectection/weights/dection_bee/yolo26s_img640_baseline_v1-nano/yolo26s_img640_baseline_v1-nano/weights/best.pt


#1. IMAGE TEST

In [4]:
IMAGE_FILE = media_source(IMAGE_PATH, "IMAGE_PATH")
IMAGE_RESULTS = {}


def run_image_test(model, model_name, image_source):
    images = collect_images(image_source)
    if not images:
        print(f"[FAIL] {model_name}: khong tim thay anh hop le trong {image_source}")
        return None
    warmup(model, images[0])
    speeds = []
    annotated = []
    labels = []
    confidences = []
    detections = 0
    passes = IMAGE_REPEATS if len(images) == 1 else 1
    for pass_index in range(passes):
        for image_path in images:
            try:
                result = model.predict(
                    source=str(image_path),
                    imgsz=IMG_SIZE,
                    conf=CONF_THRESHOLD,
                    device=DEVICE,
                    verbose=False,
                )[0]
            except Exception as error:
                print(f"[FAIL] {image_path.name}: {error}")
                continue
            speeds.append(result.speed)
            if pass_index > 0:
                continue
            detections += len(result.boxes)
            labels.extend(result.names[int(c)] for c in result.boxes.cls)
            confidences.extend(round(float(c), 3) for c in result.boxes.conf)
            annotated.append((image_path.name, result.plot()))
    if not speeds:
        print(f"[FAIL] {model_name}: khong duoc anh nao")
        return None
    inference_ms = float(np.mean([s["inference"] for s in speeds]))
    total_ms = float(np.mean([s["preprocess"] + s["inference"] + s["postprocess"] for s in speeds]))
    return {
        "model": model_name,
        "images": len(images),
        "detections": detections,
        "labels": labels,
        "confidences": confidences,
        "inference_ms": inference_ms,
        "total_ms": total_ms,
        "fps": 1000.0 / total_ms,
        "annotated": annotated,
    }


def print_image_report(report):
    print(f"[{report['model']}]")
    print(f"  images     : {report['images']}")
    print(f"  detections : {report['detections']}")
    print(f"  labels     : {', '.join(report['labels']) or '-'}")
    print(f"  confidences: {report['confidences'] or '-'}")
    print(f"  inference  : {report['inference_ms']:.2f} ms")
    print(f"  total      : {report['total_ms']:.2f} ms")
    print(f"  FPS        : {report['fps']:.1f}")


def show_annotated(title, image_bgr):
    plt.figure(figsize=(9, 6))
    plt.imshow(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.tight_layout()
    plt.show()

[FAIL] IMAGE_PATH khong ton tai: /home/loc-nguyen/PROJECT/AI/hive-management-ai/ai_engine/data/bee-dectection/testing_data/test_img.png
       cwd hien tai: /home/loc-nguyen/PROJECT/AI/hive-management-ai/ai_engine/bee-dectection/notebooks


## yolo26s_img640_baseline_v1_small

In [5]:
if IMAGE_FILE and MODEL_SMALL in MODELS:
    IMAGE_RESULTS[MODEL_SMALL] = run_image_test(MODELS[MODEL_SMALL], MODEL_SMALL, IMAGE_FILE)
    if IMAGE_RESULTS[MODEL_SMALL]:
        print_image_report(IMAGE_RESULTS[MODEL_SMALL])
        for filename, image_bgr in IMAGE_RESULTS[MODEL_SMALL]["annotated"]:
            show_annotated(f"{MODEL_SMALL} · {filename}", image_bgr)
else:
    print("[SKIP] image test cho yolo26s_img640_baseline_v1_small")

[SKIP] image test cho yolo26s_img640_baseline_v1_small


## yolo26s_img640_baseline_v1_nano

In [6]:
if IMAGE_FILE and MODEL_NANO in MODELS:
    IMAGE_RESULTS[MODEL_NANO] = run_image_test(MODELS[MODEL_NANO], MODEL_NANO, IMAGE_FILE)
    if IMAGE_RESULTS[MODEL_NANO]:
        print_image_report(IMAGE_RESULTS[MODEL_NANO])
        for filename, image_bgr in IMAGE_RESULTS[MODEL_NANO]["annotated"]:
            show_annotated(f"{MODEL_NANO} · {filename}", image_bgr)
else:
    print("[SKIP] image test cho yolo26s_img640_baseline_v1_nano")

[SKIP] image test cho yolo26s_img640_baseline_v1_nano


In [7]:
print(f"{'model':34s} {'anh':>4s} {'det':>4s} {'inf_ms':>8s} {'total_ms':>9s} {'FPS':>7s}")
for name in (MODEL_SMALL, MODEL_NANO):
    report = IMAGE_RESULTS.get(name)
    if report:
        print(
            f"{name:34s} {report['images']:4d} {report['detections']:4d} {report['inference_ms']:8.2f} "
            f"{report['total_ms']:9.2f} {report['fps']:7.1f}"
        )

model                               anh  det   inf_ms  total_ms     FPS


#2. VIDEO TEST (FPS)

In [8]:
VIDEO_FILE = media_source(VIDEO_PATH, "VIDEO_PATH", allow_dir=False)
FPS_RESULTS = []


def run_fps_test(model, model_name, video_path):
    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        print(f"[FAIL] {model_name}: khong mo duoc video: {video_path}")
        return None
    source_fps = capture.get(cv2.CAP_PROP_FPS)
    if not source_fps or source_fps <= 0:
        source_fps = 24.0
    ok, warm_frame = capture.read()
    if not ok:
        capture.release()
        print(f"[FAIL] {model_name}: khong doc duoc frame dau tien — {video_path} khong phai video hop le")
        return None
    warmup(model, warm_frame)
    output_path = video_path.with_name(f"{video_path.stem}_predicted_{model_name}.mp4")
    writer = None
    frame_fps = []
    sample_frames = []
    detections = 0
    started = time.perf_counter()
    while len(frame_fps) < FPS_MAX_FRAMES:
        ok, frame = capture.read()
        if not ok:
            break
        tick = time.perf_counter()
        result = model.predict(
            source=frame,
            imgsz=IMG_SIZE,
            conf=CONF_THRESHOLD,
            device=DEVICE,
            verbose=False,
        )[0]
        frame_fps.append(1.0 / (time.perf_counter() - tick))
        detections += len(result.boxes)
        annotated = result.plot()
        if writer is None:
            height, width = annotated.shape[:2]
            writer = cv2.VideoWriter(str(output_path), cv2.VideoWriter_fourcc(*"mp4v"), source_fps, (width, height))
        writer.write(annotated)
        if (len(frame_fps) - 1) % SAMPLE_FRAME_STRIDE == 0 and len(sample_frames) < MAX_SAMPLE_FRAMES:
            sample_frames.append((len(frame_fps), annotated))
    capture.release()
    if writer is not None:
        writer.release()
    if not frame_fps:
        print(f"[FAIL] {model_name}: khong co frame video nao — kiem tra dinh dang: {video_path}")
        return None
    if not output_path.is_file() or output_path.stat().st_size == 0:
        print(f"[FAIL] khong ghi duoc video du doan: {output_path}")
        output_path = None
    else:
        print(f"[PASS] video du doan -> {output_path}")
    values = np.array(frame_fps)
    return {
        "model": model_name,
        "frames": len(frame_fps),
        "detections": detections,
        "duration_s": time.perf_counter() - started,
        "frame_fps": frame_fps,
        "mean_fps": float(values.mean()),
        "median_fps": float(np.median(values)),
        "min_fps": float(values.min()),
        "max_fps": float(values.max()),
        "output_path": output_path,
        "sample_frames": sample_frames,
    }


def print_fps_report(report):
    print(f"[{report['model']}]")
    print(f"  frames     : {report['frames']}")
    print(f"  detections : {report['detections']}")
    print(f"  mean FPS   : {report['mean_fps']:.1f}")
    print(f"  median FPS : {report['median_fps']:.1f}")
    print(f"  min / max  : {report['min_fps']:.1f} / {report['max_fps']:.1f}")
    print(f"  duration   : {report['duration_s']:.2f} s")
    print(f"  predicted  : {report['output_path'] or '-'}")


def plot_fps_chart(report):
    values = np.array(report["frame_fps"])
    mean_fps = report["mean_fps"]
    plt.figure(figsize=(10, 4))
    plt.plot(np.arange(1, len(values) + 1), values, marker="o", markersize=3, linewidth=1.2, label="FPS moi frame")
    plt.axhline(mean_fps, linestyle="--", linewidth=1.2, color="crimson", label=f"FPS binh jun: {mean_fps:.1f}")
    plt.title(f"FPS test - {report['model']}")
    plt.xlabel("Frame")
    plt.ylabel("FPS")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


def show_sample_frames(report):
    frames = report["sample_frames"]
    if not frames:
        return
    columns = min(3, len(frames))
    rows = (len(frames) + columns - 1) // columns
    figure, axes = plt.subplots(rows, columns, figsize=(5 * columns, 3.5 * rows), squeeze=False)
    flat = axes.ravel()
    for axis, (index, image_bgr) in zip(flat, frames):
        axis.imshow(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
        axis.set_title(f"{report['model']} · frame {index}")
        axis.axis("off")
    for axis in flat[len(frames):]:
        axis.axis("off")
    figure.tight_layout()
    plt.show()

[FAIL] VIDEO_PATH khong ton tai: /home/loc-nguyen/PROJECT/AI/hive-management-ai/ai_engine/bee-dectection/notebooks/stock-footage-close-up-of-honeybees-warming-in-sun-on-their-beehive-beekeeping-process-honey-productio-and.mp4
       cwd hien tai: /home/loc-nguyen/PROJECT/AI/hive-management-ai/ai_engine/bee-dectection/notebooks


## yolo26s_img640_baseline_v1_small

In [9]:
if VIDEO_FILE and MODEL_SMALL in MODELS:
    small_fps = run_fps_test(MODELS[MODEL_SMALL], MODEL_SMALL, VIDEO_FILE)
    if small_fps:
        FPS_RESULTS.append(small_fps)
        print_fps_report(small_fps)
        plot_fps_chart(small_fps)
        show_sample_frames(small_fps)
else:
    print("[SKIP] FPS test cho yolo26s_img640_baseline_v1_small")

[SKIP] FPS test cho yolo26s_img640_baseline_v1_small


## yolo26s_img640_baseline_v1_nano

In [10]:
if VIDEO_FILE and MODEL_NANO in MODELS:
    nano_fps = run_fps_test(MODELS[MODEL_NANO], MODEL_NANO, VIDEO_FILE)
    if nano_fps:
        FPS_RESULTS.append(nano_fps)
        print_fps_report(nano_fps)
        plot_fps_chart(nano_fps)
        show_sample_frames(nano_fps)
else:
    print("[SKIP] FPS test cho yolo26s_img640_baseline_v1_nano")

[SKIP] FPS test cho yolo26s_img640_baseline_v1_nano


In [11]:
print(f"{'model':34s} {'frames':>6s} {'det':>5s} {'mean':>7s} {'median':>7s} {'min':>7s} {'max':>7s}")
for report in FPS_RESULTS:
    print(
        f"{report['model']:34s} {report['frames']:6d} {report['detections']:5d} "
        f"{report['mean_fps']:7.1f} {report['median_fps']:7.1f} "
        f"{report['min_fps']:7.1f} {report['max_fps']:7.1f}"
    )

model                              frames   det    mean  median     min     max


In [18]:
import cv2
from ultralytics import YOLO

model_path = r"/home/loc-nguyen/PROJECT/AI/hive-management-ai/ai_engine/bee-dectection/weights/dection_bee/yolo26s_img640_baseline_v1_small/yolo26s_img640_baseline_v1_small/weights/best.pt"
input_video =r"/home/loc-nguyen/PROJECT/AI/hive-management-ai/0_DEMO_VIDEO/DEMO_2.mp4"
model = YOLO(model_path)
cap = cv2.VideoCapture(input_video)

if not cap.isOpened():
    raise RuntimeError(f"Không mở được video: {input_video}")

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        results = model(frame, verbose=False, conf = 0.6)

        # Vẽ bounding box / nhãn lên frame để hiển thị
        annotated_frame = results[0].plot()
        cv2.imshow("YOLO Video", annotated_frame)

        # Chờ 1ms và kiểm tra phím 'q' (mặt nạ 0xFF đảm bảo tương thích đa nền tảng)
        if cv2.waitKey(20) & 0xFF == ord("q"):
            break
finally:
    # Luôn giải phóng tài nguyên và đóng cửa sổ khi kết thúc hoặc bấm thoát
    cap.release()
    cv2.destroyAllWindows()